# 04 — Evaluation & Model Comparison

**Metrics:** RMSE, MAE, Precision@K, Recall@K (K=10)  
**Models:** Global Average, User-Based CF, Item-Based CF, SVD

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED_DATA_DIR = '../data/processed/'
RESULTS_DIR        = '../results/'
FIGURES_DIR        = '../results/figures/'

os.makedirs(FIGURES_DIR, exist_ok=True)
print('Imports successful.')

---
## 2. Load Results & Plot Comparison

In [ ]:
results = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics_all.csv'))
print(results.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = sns.color_palette('muted', len(results))

# RMSE bar chart
bars1 = axes[0].bar(results['Model'], results['RMSE'], color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars1, results['RMSE']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[0].set_title('RMSE by Model (lower is better)', fontsize=13)
axes[0].set_ylabel('RMSE', fontsize=11)
axes[0].set_ylim(0, results['RMSE'].max() * 1.15)
axes[0].tick_params(axis='x', rotation=15)
sns.despine(ax=axes[0])

# MAE bar chart
bars2 = axes[1].bar(results['Model'], results['MAE'], color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars2, results['MAE']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)
axes[1].set_title('MAE by Model (lower is better)', fontsize=13)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_ylim(0, results['MAE'].max() * 1.15)
axes[1].tick_params(axis='x', rotation=15)
sns.despine(ax=axes[1])

plt.suptitle('Model Comparison — CiaoDVD Recommendation System', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png')

---
## 3. Bonus — Precision@K and Recall@K (K=10)

Precision@K: of the top-K movies recommended, what fraction did the user actually like?  
Recall@K: of all movies the user actually liked, what fraction appeared in the top-K?

A rating >= 4.0 is treated as "relevant" (liked).

In [ ]:
# Rebuild data and models needed for top-K evaluation
ratings = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'ratings_clean.csv'))

train_df, test_df = train_test_split(ratings, test_size=0.20, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

global_mean = train_df['rating'].mean()
RELEVANCE_THRESHOLD = 4.0
K = 10

print(f'Relevance threshold : >= {RELEVANCE_THRESHOLD}')
print(f'K                   : {K}')

In [ ]:
def precision_recall_at_k(user_predictions, user_actuals, k, threshold):
    """
    Given predicted and actual ratings for one user:
    - Sort predictions descending, take top K
    - Precision@K = relevant items in top-K / K
    - Recall@K    = relevant items in top-K / total relevant items
    """
    # Sort by predicted rating descending
    sorted_idx      = np.argsort(user_predictions)[::-1][:k]
    top_k_actuals   = np.array(user_actuals)[sorted_idx]
    top_k_relevant  = np.sum(top_k_actuals >= threshold)
    total_relevant  = np.sum(np.array(user_actuals) >= threshold)

    precision = top_k_relevant / k
    recall    = top_k_relevant / total_relevant if total_relevant > 0 else 0.0
    return precision, recall

print('Precision/Recall@K function defined.')

### 3.1 Rebuild SVD Predictions

In [ ]:
user_item_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
user_means       = user_item_matrix.mean(axis=1)
matrix_demeaned  = user_item_matrix.subtract(user_means, axis=0).fillna(0)

best_n = 10  # best k found in 03_svd_model.ipynb
U, sigma, Vt = svds(matrix_demeaned.values.astype(float), k=best_n)
predicted    = np.dot(np.dot(U, np.diag(sigma)), Vt) + user_means.values.reshape(-1, 1)
svd_pred_matrix = pd.DataFrame(predicted, index=matrix_demeaned.index, columns=matrix_demeaned.columns)

print('SVD prediction matrix rebuilt.')

### 3.2 Rebuild Item-Based CF Predictions

In [ ]:
user_item_filled = user_item_matrix.fillna(0)
item_similarity  = cosine_similarity(user_item_filled.T)
item_sim_df      = pd.DataFrame(item_similarity, index=user_item_filled.columns, columns=user_item_filled.columns)

def predict_item_based(user_id, movie_id, item_sim_df, user_item_matrix, global_mean, k=20):
    if user_id not in user_item_matrix.index or movie_id not in item_sim_df.index:
        return global_mean
    sim_scores   = item_sim_df[movie_id].drop(index=movie_id)
    user_ratings = user_item_matrix.loc[user_id]
    sim_scores   = sim_scores[user_ratings > 0]
    if sim_scores.empty or sim_scores.sum() == 0:
        return global_mean
    top_k            = sim_scores.nlargest(k)
    neighbor_ratings = user_item_matrix.loc[user_id, top_k.index].values
    return np.dot(top_k.values, neighbor_ratings) / np.sum(np.abs(top_k.values))

print('Item-based CF function ready.')

### 3.3 Compute Precision@K and Recall@K per User

In [ ]:
# Group test set by user — only evaluate users with enough test ratings
test_grouped = test_df.groupby('userId')

svd_precisions, svd_recalls   = [], []
item_precisions, item_recalls = [], []
global_precisions, global_recalls = [], []

for user_id, group in test_grouped:
    if len(group) < K:
        continue  # skip users with fewer test ratings than K

    actuals   = group['rating'].tolist()
    movie_ids = group['movieId'].tolist()

    # SVD predictions
    svd_preds = [
        svd_pred_matrix.loc[user_id, mid]
        if user_id in svd_pred_matrix.index and mid in svd_pred_matrix.columns
        else global_mean
        for mid in movie_ids
    ]

    # Item-Based CF predictions
    item_preds = [
        predict_item_based(user_id, mid, item_sim_df, user_item_filled, global_mean)
        for mid in movie_ids
    ]

    # Global average predictions
    global_preds = [global_mean] * len(actuals)

    p_svd,  r_svd  = precision_recall_at_k(svd_preds,    actuals, K, RELEVANCE_THRESHOLD)
    p_item, r_item = precision_recall_at_k(item_preds,   actuals, K, RELEVANCE_THRESHOLD)
    p_glob, r_glob = precision_recall_at_k(global_preds, actuals, K, RELEVANCE_THRESHOLD)

    svd_precisions.append(p_svd);   svd_recalls.append(r_svd)
    item_precisions.append(p_item); item_recalls.append(r_item)
    global_precisions.append(p_glob); global_recalls.append(r_glob)

print(f'Users evaluated: {len(svd_precisions)}')

In [ ]:
topk_results = pd.DataFrame({
    'Model':       ['Global Average', 'Item-Based CF', f'SVD (k={best_n})'],
    f'Precision@{K}': [
        round(np.mean(global_precisions), 4),
        round(np.mean(item_precisions),   4),
        round(np.mean(svd_precisions),    4)
    ],
    f'Recall@{K}': [
        round(np.mean(global_recalls), 4),
        round(np.mean(item_recalls),   4),
        round(np.mean(svd_recalls),    4)
    ]
})

print(topk_results.to_string(index=False))
topk_results.to_csv(os.path.join(RESULTS_DIR, 'metrics_topk.csv'), index=False)
print('\nSaved metrics_topk.csv')

---
## 4. Discussion

*(Fill in after running — answer these questions in your own words)*

**Which model performed best and why?**

SVD achieved the lowest RMSE (0.7235) and MAE (0.5654), outperforming all other models. This is expected because SVD learns latent factors that capture hidden patterns in the data — it effectively compresses the sparse rating matrix into a dense representation of user preferences and movie attributes. Unlike CF which relies on direct rating overlap between users or items, SVD generalizes better across the sparse CiaoDVD dataset.

Item-Based CF came in a close second (RMSE 0.7292), suggesting that movie-to-movie similarity is more stable than user-to-user similarity in this dataset. User-Based CF performed no better than the global average baseline, likely because most users have rated very few movies (median = 15), making it hard to find reliable neighbors.

**What do the Precision@K and Recall@K results tell us?**

*(Write your observation here after running — which model ranks relevant items highest?)*